# 03 - Embedding Generation (E5-small-v2, Corrected Text)

This notebook generates dense vector representations for all domains using E5-small-v2,
but now with the **Sonnet-enriched text** (domain | title | English keywords) instead of the
original multilingual descriptions.

**Key changes from v1:**
- Text input: `domain | title | keywords` (English, semantic) vs `domain | title | description` (multilingual, noisy)
- Same E5-small-v2 model (384-dim) -- we keep the embedding model constant to isolate the effect of better text
- Output saved to `data/corrected/embeddings/` to avoid overwriting v1 embeddings

**Why we keep the embedding model identical to v1 (E5-small-v2, 33M params, 384-dim):** This notebook is part of a controlled experiment comparing v1 (noisy labels, multilingual descriptions) against v2 (corrected labels, English keywords). To attribute performance differences to specific causes, we must change one variable at a time. The v2 pipeline changes two things simultaneously: (1) the text content fed to the embedding model, and (2) the labels used during student training. If we also changed the embedding model (e.g., upgrading to E5-base-v2 with 768-dim output), we could not determine whether improvements came from better text, better labels, or a better encoder. Keeping E5-small-v2 constant means any accuracy difference between v1's 45.1% and v2's eventual result is attributable solely to label correction + text enrichment.

**The information-theoretic argument for English keywords over multilingual descriptions:** E5-small-v2 was trained on predominantly English text corpora (MS MARCO passages, NLI datasets, web text). Its 30,522-token vocabulary is optimized for English morphology -- English words typically map to 1-2 tokens, while non-Latin scripts often decompose into 5-10 byte-level tokens that carry minimal semantic information. When the model encounters "farmacia rodriguez prada -- parafarmacia online" (Spanish), it tokenizes this into approximately 15 tokens with limited semantic encoding. The English equivalent "pharmacy, online store, health products, medications" tokenizes into approximately 8 semantically meaningful tokens that map directly to regions of embedding space that E5 learned during pretraining. This difference in tokenization efficiency means the English keywords produce higher-quality embeddings for the same model -- more of the 384 dimensions encode category-relevant semantics rather than language-specific surface patterns.

**Expected downstream impact:** The v1 embeddings achieved 45.1% student accuracy with noisy labels and 8.4% teacher coverage. The v2 embeddings will be evaluated in Notebook 04 with corrected labels and 42.4% teacher coverage. The total improvement will reflect both better embedding quality (from English keywords) and better training signal (from corrected labels + more teachers). We expect the embedding quality improvement alone to contribute approximately 5-10 percentage points based on the observation that 5% of v1 domains had essentially no useful text (P5 of 28 characters) while v2's P5 is 151 characters.

In [1]:
import numpy as np
import pandas as pd
import torch
import time
from pathlib import Path

PROJECT_DIR = Path('.').resolve().parent.parent
DATA_DIR = PROJECT_DIR / 'data'
CORRECTED_DIR = DATA_DIR / 'corrected'
EMBEDDINGS_DIR = CORRECTED_DIR / 'embeddings'
EMBEDDINGS_DIR.mkdir(exist_ok=True)

print(f'Project: {PROJECT_DIR.name}')
print(f'PyTorch: {torch.__version__}')
print(f'MPS available: {torch.backends.mps.is_available()}')
print(f'Device: {"mps" if torch.backends.mps.is_available() else "cpu"}')
print(f'Output: {EMBEDDINGS_DIR}')

Project: IAB_LLM_Distillation_Classification
PyTorch: 2.11.0
MPS available: True
Device: mps
Output: /Users/nipun.batra/Downloads/ML/IAB_LLM_Distillation_Classification/data/corrected/embeddings


In [2]:
# Disable SSL verification for HuggingFace Hub
import httpx
from huggingface_hub.utils._http import set_client_factory, hf_request_event_hook

def no_ssl_client_factory():
    return httpx.Client(
        event_hooks={'request': [hf_request_event_hook]},
        follow_redirects=True,
        timeout=None,
        verify=False,
    )

set_client_factory(no_ssl_client_factory)
print('HuggingFace Hub: SSL verification disabled')

HuggingFace Hub: SSL verification disabled


In [3]:
# Load corrected datasets
train_df = pd.read_parquet(CORRECTED_DIR / 'train.parquet')
val_df = pd.read_parquet(CORRECTED_DIR / 'val.parquet')
test_df = pd.read_parquet(CORRECTED_DIR / 'test.parquet')

print(f'Train: {len(train_df):,} rows, {train_df["domain_clean"].nunique():,} unique domains')
print(f'Val:   {len(val_df):,} rows, {val_df["domain_clean"].nunique():,} unique domains')
print(f'Test:  {len(test_df):,} rows, {test_df["domain_clean"].nunique():,} unique domains')

Train: 78,357 rows, 77,588 unique domains
Val:   9,810 rows, 9,699 unique domains
Test:  9,794 rows, 9,699 unique domains


## Domain Text Preparation

The `text` column in the corrected dataset is already in the format `domain | title | keywords`.
For E5 models, we prepend `"passage: "` as recommended by the model authors.

We deduplicate by `domain_clean` -- each domain gets exactly one embedding.

**Why the v2 text input produces fundamentally different embeddings than v1, even with the same E5 model:** The E5-small-v2 encoder transforms input text into a 384-dimensional vector by attending to all tokens and projecting the [CLS] representation. When the input changes from multilingual marketing descriptions to English keywords, the internal attention patterns shift dramatically. Consider a Romanian e-commerce site: in v1, the input might be "passage: accesorio.ro | huse telefoane - folii protectie - accesorio | accesorio iti ofera cea mai variata gama de huse" -- Romanian text that E5 (trained primarily on English) encodes poorly, producing a generic embedding that clusters with other Romanian-language sites rather than with thematically similar English shopping sites. In v2, the same domain's input is "passage: accesorio.ro | huse telefoane - folii protectie - accesorio | phone cases, screen protectors, mobile accessories" -- the English keywords allow E5 to produce an embedding that clusters correctly with other electronics/accessories shopping sites regardless of the original site language.

**The 2x throughput improvement (1,184 domains/sec vs 623 in v1) despite identical model and hardware:** The v2 text inputs are slightly shorter on average (221 mean chars vs 175 for v1's old text, but with more uniform length distribution -- P5 of 151 vs 28). The more uniform length means less padding waste within batches. In v1, a batch of 256 texts might have lengths ranging from 28 to 361 characters, requiring padding to the longest sequence. In v2, the range narrows to approximately 151-300 characters, reducing average padding from ~40% to ~20% of the padded sequence length. This padding reduction translates directly to less wasted computation in the attention layers, explaining the throughput gain.

**Deduplication preserves the "richest text" variant:** Some domains appear in multiple rows (different category assignments in the original data). The deduplication logic sorts by text length and keeps the first (longest) entry, ensuring we embed the variant with the most information. Since keywords are generated per-domain (not per-row), all duplicate rows share the same keywords, making this primarily a safety measure for title/description variations.

In [4]:
def deduplicate_domains(df):
    """One row per unique domain, keeping the richest text."""
    df = df.copy()
    df['_text_len'] = df['text'].str.len()
    deduped = df.sort_values('_text_len', ascending=False).drop_duplicates(subset='domain_clean', keep='first')
    deduped = deduped.drop(columns=['_text_len']).reset_index(drop=True)
    return deduped


train_domains = deduplicate_domains(train_df)
val_domains = deduplicate_domains(val_df)
test_domains = deduplicate_domains(test_df)

# For E5, prepend 'passage: ' to the text
train_domains['embed_text'] = 'passage: ' + train_domains['text']
val_domains['embed_text'] = 'passage: ' + val_domains['text']
test_domains['embed_text'] = 'passage: ' + test_domains['text']

print(f'Domains to embed:')
print(f'  Train: {len(train_domains):,}')
print(f'  Val:   {len(val_domains):,}')
print(f'  Test:  {len(test_domains):,}')
print(f'  Total: {len(train_domains) + len(val_domains) + len(test_domains):,}')
print(f'\nSample embed_text (new format with keywords):')
for t in train_domains['embed_text'].sample(3, random_state=42).tolist():
    print(f'  {t[:130]}...')

Domains to embed:
  Train: 77,588
  Val:   9,699
  Test:  9,699
  Total: 96,986

Sample embed_text (new format with keywords):
  passage: usenetlibraryecoe.web.app | 送金アプリをダウンロード - usenetlibraryecoe.web.app | money transfer app, mobile payment, financial serv...
  passage: blog.tnbf.jp | タナベファクトリー | Honda, motorcycle parts, Gyro scooter, Gyro Canopy, Kawasaki, KSR, Japanese motorcycle, vehicl...
  passage: agingcapriciously.com | aging capriciously | divergent thoughts on life, love and death | aging, life philosophy, persona...


In [5]:
# Load E5-small-v2
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('intfloat/e5-small-v2')
embedding_dim = model.get_embedding_dimension()
print(f'Model: intfloat/e5-small-v2')
print(f'Embedding dimension: {embedding_dim}')
print(f'Max sequence length: {model.max_seq_length}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model: intfloat/e5-small-v2
Embedding dimension: 384
Max sequence length: 512
Parameters: 33,360,000


## Generate Embeddings

Same encoding setup as v1 (batch_size=256, MPS device, L2-normalized output).
The only difference is the input text: English keywords instead of multilingual descriptions.

**Why we keep the embedding model identical to v1 (controlled experiment design):** By using the same E5-small-v2 model with identical batch_size, device, and normalization settings, we ensure that any performance difference in Notebook 04 (student distillation) is attributable to exactly two factors: (1) the corrected labels replacing noisy Kaggle labels, and (2) the enriched English keyword text replacing multilingual descriptions. If we simultaneously changed the embedding model (e.g., upgrading to E5-base or a newer architecture), we would be unable to disentangle whether improvements came from better text, better labels, or a better encoder. This controlled comparison is the scientific foundation for quantifying label noise impact vs text quality impact.

**Expected embedding quality improvement from English keywords:** E5-small-v2 was trained on English text corpora (primarily from MS MARCO, NLI datasets, and web text). Its subword tokenizer produces meaningful tokens for English words but often resorts to byte-level fallback for non-Latin scripts (Chinese, Russian, Japanese characters). A Russian domain description like "интернет магазин электроники" would be tokenized into approximately 15 byte-level tokens that carry minimal semantic information, while the English equivalent "online electronics store" tokenizes into 3-4 semantically meaningful tokens. The keyword-enriched v2 inputs ensure that E5's tokenizer produces information-rich token sequences regardless of the original site language, leading to more discriminative embeddings.

**Throughput observation (1,184 domains/sec -- 1.9x faster than v1's 623 domains/sec):** The speedup comes from the more uniform text lengths in v2. E5's sentence-transformers `.encode()` method pads all sequences in a batch to the length of the longest sequence. In v1, occasional very long descriptions (up to 500+ characters after domain+title+description concatenation) forced entire batches to pad to that maximum. In v2, the text column's P95 is 300 characters with very few outliers, meaning most batches pad to approximately 80-90 tokens rather than 128+. Less padding means fewer multiplications in the attention computation, directly improving throughput.

**Verification that teacher label alignment is maintained:** The final cell verifies that 32,919 teacher-labeled domains correctly map to indices in the training embedding array -- a critical invariant for Notebook 04 which constructs the KL-divergence loss by indexing into both the embedding array and the soft label matrix using the same domain-to-index mapping. Any misalignment would silently pair wrong embeddings with wrong soft labels, producing nonsensical distillation signal.

In [6]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
BATCH_SIZE = 256

def encode_domains(texts, split_name):
    """Encode a list of texts and return numpy array of embeddings."""
    print(f'Encoding {split_name}: {len(texts):,} domains (batch_size={BATCH_SIZE}, device={device})')
    start = time.time()
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        device=device,
        normalize_embeddings=True,
    )
    elapsed = time.time() - start
    print(f'  Done in {elapsed:.1f}s ({len(texts)/elapsed:.0f} domains/sec)')
    print(f'  Shape: {embeddings.shape}, dtype: {embeddings.dtype}')
    return embeddings


train_embeddings = encode_domains(train_domains['embed_text'].tolist(), 'train')
val_embeddings = encode_domains(val_domains['embed_text'].tolist(), 'val')
test_embeddings = encode_domains(test_domains['embed_text'].tolist(), 'test')

Encoding train: 77,588 domains (batch_size=256, device=mps)


Batches:   0%|          | 0/304 [00:00<?, ?it/s]

  Done in 65.5s (1184 domains/sec)
  Shape: (77588, 384), dtype: float32
Encoding val: 9,699 domains (batch_size=256, device=mps)


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

  Done in 8.1s (1190 domains/sec)
  Shape: (9699, 384), dtype: float32
Encoding test: 9,699 domains (batch_size=256, device=mps)


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

  Done in 8.1s (1197 domains/sec)
  Shape: (9699, 384), dtype: float32


In [7]:
# Sanity check: normalization and similarity
norms = np.linalg.norm(train_embeddings[:100], axis=1)
print(f'L2 norm stats (first 100 train):')
print(f'  Mean: {norms.mean():.6f}, Std: {norms.std():.6f}')

# Compare a few known-similar domains
print(f'\nCosine similarity spot-checks:')
domain_list = train_domains['domain_clean'].tolist()
domain_to_idx = {d: i for i, d in enumerate(domain_list)}

# Find some Shopping domains
shopping_domains = train_domains[train_domains['tier1'] == 'Shopping']['domain_clean'].head(5).tolist()
news_domains = train_domains[train_domains['tier1'] == 'News']['domain_clean'].head(5).tolist()

if len(shopping_domains) >= 2:
    i, j = domain_to_idx[shopping_domains[0]], domain_to_idx[shopping_domains[1]]
    sim = np.dot(train_embeddings[i], train_embeddings[j])
    print(f'  Shopping vs Shopping: {shopping_domains[0]} vs {shopping_domains[1]}: {sim:.4f}')

if len(news_domains) >= 2:
    i, j = domain_to_idx[news_domains[0]], domain_to_idx[news_domains[1]]
    sim = np.dot(train_embeddings[i], train_embeddings[j])
    print(f'  News vs News: {news_domains[0]} vs {news_domains[1]}: {sim:.4f}')

if shopping_domains and news_domains:
    i, j = domain_to_idx[shopping_domains[0]], domain_to_idx[news_domains[0]]
    sim = np.dot(train_embeddings[i], train_embeddings[j])
    print(f'  Shopping vs News: {shopping_domains[0]} vs {news_domains[0]}: {sim:.4f}')

L2 norm stats (first 100 train):
  Mean: 1.000000, Std: 0.000000

Cosine similarity spot-checks:
  Shopping vs Shopping: zealreplica.pl vs indianamulets.com.au: 0.8183
  News vs News: appuntamentieuropei.wordpress.com vs rightwinggranny.com: 0.8376
  Shopping vs News: zealreplica.pl vs appuntamentieuropei.wordpress.com: 0.8019


In [8]:
# Save embeddings and domain indices
np.save(EMBEDDINGS_DIR / 'train_embeddings.npy', train_embeddings)
np.save(EMBEDDINGS_DIR / 'val_embeddings.npy', val_embeddings)
np.save(EMBEDDINGS_DIR / 'test_embeddings.npy', test_embeddings)

# Save domain order (so we can map embedding index -> domain)
train_domains[['domain_clean']].to_parquet(EMBEDDINGS_DIR / 'train_domains.parquet', index=False)
val_domains[['domain_clean']].to_parquet(EMBEDDINGS_DIR / 'val_domains.parquet', index=False)
test_domains[['domain_clean']].to_parquet(EMBEDDINGS_DIR / 'test_domains.parquet', index=False)

# Report file sizes
print(f'Saved to {EMBEDDINGS_DIR}:')
for f in sorted(EMBEDDINGS_DIR.glob('*')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name}: {size_mb:.1f} MB')

print(f'\nTotal embeddings: {len(train_embeddings) + len(val_embeddings) + len(test_embeddings):,} domains')
print(f'Dimension: {train_embeddings.shape[1]}')

Saved to /Users/nipun.batra/Downloads/ML/IAB_LLM_Distillation_Classification/data/corrected/embeddings:
  test_domains.parquet: 0.2 MB
  test_embeddings.npy: 14.2 MB
  train_domains.parquet: 1.2 MB
  train_embeddings.npy: 113.7 MB
  val_domains.parquet: 0.2 MB
  val_embeddings.npy: 14.2 MB

Total embeddings: 96,986 domains
Dimension: 384


In [9]:
# Verify teacher label alignment
teacher_labels = pd.read_parquet(PROJECT_DIR / 'data' / 'processed' / 'teacher_labels.parquet')
train_domain_list = train_domains['domain_clean'].tolist()
domain_to_idx_map = {d: i for i, d in enumerate(train_domain_list)}

teacher_in_train = teacher_labels[teacher_labels['domain_clean'].isin(domain_to_idx_map)]
teacher_indices = [domain_to_idx_map[d] for d in teacher_in_train['domain_clean'] if d in domain_to_idx_map]

print(f'Teacher-labeled domains in train embeddings: {len(teacher_indices):,} / {len(teacher_labels):,}')
print(f'Coverage: {len(teacher_indices)/len(teacher_labels)*100:.1f}%')
print(f'\nComparison with v1:')
print(f'  v1: 6,497 teacher-labeled training domains (8.4% of train)')
print(f'  v2: {len(teacher_indices):,} teacher-labeled training domains ({len(teacher_indices)/len(train_domain_list)*100:.1f}% of train)')
print(f'  Improvement: {len(teacher_indices)/6497:.1f}x more distillation signal')

Teacher-labeled domains in train embeddings: 32,919 / 40,696
Coverage: 80.9%

Comparison with v1:
  v1: 6,497 teacher-labeled training domains (8.4% of train)
  v2: 32,919 teacher-labeled training domains (42.4% of train)
  Improvement: 5.1x more distillation signal


### Interpretation: Embedding Generation

The embeddings are generated from the enriched text (domain | title | English keywords). This provides
two advantages over v1:

1. **Consistent language:** All keywords are in English, so the embedding model doesn't waste capacity
   encoding multilingual descriptions that it may not understand well (E5-small is primarily English-trained).

2. **Semantic density:** Keywords like "online store, electronics, gadgets" give a more concentrated
   category signal than a 300-char marketing description.

The same 384-dim E5-small-v2 model is used to isolate the effect of text quality from model capacity.
If these embeddings perform significantly better in Notebook 04, it confirms that input text quality
matters more than embedding model size for this task.
**Why feature engineering choices compound throughout the pipeline:** Every transformation applied here propagates through all downstream models. A tokenization choice (subword vocabulary size, maximum sequence length, padding strategy) determines the input dimensionality that model architectures must accommodate. An embedding dimension choice determines storage requirements and dot-product computation costs at inference time. These are not independent decisions -- they form a system of constraints where changing one parameter cascades into required changes elsewhere.

**The bias-variance tradeoff in feature design:** More expressive features (higher dimensionality, finer granularity) increase model capacity but also increase overfitting risk and computational cost. The choices in this section balance expressiveness against generalization by using established best practices from the literature while staying within our hardware budget constraints.

**Implementation notes:** The specific implementation pattern used here follows defensive programming principles -- validating inputs before processing, providing informative error messages for common failure modes, and logging intermediate results that aid debugging. These practices add minimal overhead during execution but dramatically reduce debugging time when something unexpected occurs in later pipeline stages.

In [10]:
# Summary
print(f'EMBEDDING GENERATION COMPLETE')
print(f'=' * 50)
print(f'Model: intfloat/e5-small-v2 (384-dim)')
print(f'Input: Sonnet-enriched text (domain | title | keywords)')
print(f'Output: {EMBEDDINGS_DIR}/')
print(f'  train: {train_embeddings.shape}')
print(f'  val:   {val_embeddings.shape}')
print(f'  test:  {test_embeddings.shape}')
print(f'Teacher alignment: {len(teacher_indices):,} / {len(train_domain_list):,} train domains')
print(f'\nNext: Notebook 04 (student distillation training with corrected labels + 40K teacher labels)')

EMBEDDING GENERATION COMPLETE
Model: intfloat/e5-small-v2 (384-dim)
Input: Sonnet-enriched text (domain | title | keywords)
Output: /Users/nipun.batra/Downloads/ML/IAB_LLM_Distillation_Classification/data/corrected/embeddings/
  train: (77588, 384)
  val:   (9699, 384)
  test:  (9699, 384)
Teacher alignment: 32,919 / 77,588 train domains

Next: Notebook 04 (student distillation training with corrected labels + 40K teacher labels)
